# Colab 11 — Real CATH AA, normLev, V1 + V2

**Goal.** Train the colab10 architecture on real CATH-S20 amino-acid sequences with the supervisor's precomputed `aa_score` labels, plus synthetic perturbation pairs to populate the high-similarity range that S20 is too redundancy-reduced to provide. Evaluate with V1 (pointwise) and V2 (same/different superfamily separation).

## What changed from colab10

| | colab10 (synthetic) | colab11 (real CATH) |
|---|---|---|
| Data | uniform random AA strings | CATH-S20 train70/test30, pre-supplied by supervisor |
| Length | 40–80 (synthetic), MAX_LEN=96 | 50–200 (filtered), MAX_LEN=200 |
| Pair source | 80% perturbation + 20% random, all synthetic | 30K natural pairs (`aa_score` from `cath_s20_pairs_sample.csv.gz`) + 30K synthetic perturbation pairs |
| Label | computed `1 − lev/max(L)` on synthetic seqs | `aa_score` for natural pairs, computed `normLev` for synthetic pairs |
| V1 | one scatter | two scatters: natural held-out (real biology) + synthetic held-out (full-range calibration) |
| V2 | not implemented (no superfamily on synthetic) | random test30 pair sample, histogram + AUROC on same vs different `SuperFamily` |
| `torch.norm` | deprecated form used | replaced with `torch.linalg.vector_norm` |

## Why mixed natural + synthetic

CATH-S20 is sequence-redundancy-reduced (≤20% identity). The natural `aa_score` distribution is heavily concentrated in [0.1, 0.3] — out of 111M pairs, only 6,765 have `aa_score ≥ 0.3` and just 6 are ≥0.7. Training on natural pairs alone would let the network minimize MSE by predicting a constant ~0.2.

Synthetic perturbation pairs (real CATH protein + k random sub/ins/del → measure actual normLev) populate the [0.3, 1.0] range we need for the model to learn the function across its full domain. Biological success is then judged by V2 on natural pairs.

## 1. Setup

Same `rm -rf` + clone pattern as previous colabs. Data lives in the repo at `sampledata/cath/` — pulled down with the clone, no Drive mount needed.

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'

# Sanity check: list the four expected files
import os
expected = ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz',
            'cath_s20_pairs_sample.csv.gz', 'cath_s20_pairs_high.csv.gz']
for f in expected:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK " if os.path.exists(p) else "MISSING ":<10} {p}')

In [ ]:
!pip install torch torchvision --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import roc_auc_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Constants — alphabet, lengths

Same vocab as colab10 (20 standard AA + `<PAD>` = 21). MAX_LEN raised from 96 to 200; length filter band 50–200 covers ~70% of train70 proteins (decided from server-side length distribution scan).

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
VOCAB_SIZE = len(AA_ALPHABET) + 1   # +1 for <PAD>
PAD_IDX = len(AA_ALPHABET)          # PAD index 20
AA_SET = set(AA_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

MIN_LEN_KEEP = 50
MAX_LEN_KEEP = 200
MAX_LEN = 200   # padding length

print(f'Vocab size (incl. PAD): {VOCAB_SIZE}, PAD index: {PAD_IDX}')
print(f'Length filter: [{MIN_LEN_KEEP}, {MAX_LEN_KEEP}], padded to {MAX_LEN}')

## 3. Levenshtein, normLev, encoding helpers

Same Levenshtein DP as colab10. `is_standard` filters out proteins containing non-standard residues (X = unknown, U = selenocysteine, B/Z = ambiguous, etc.) — keeping the alphabet clean.

In [ ]:
def levenshtein(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]

def norm_lev(a, b):
    L = max(len(a), len(b))
    if L == 0:
        return 1.0
    return 1.0 - levenshtein(a, b) / L

def is_standard(seq):
    return all(c in AA_SET for c in seq)

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    idx = [CHAR_TO_IDX[c] for c in seq][:max_len]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

# Sanity: identical seq → 1.0; sub'd seq → 0.95 etc.
s = 'ACDEFGHIKLMNPQRSTVWY'
print(f"normLev(s,s)            = {norm_lev(s, s):.3f}  (expect 1.0)")
print(f"normLev(s, s+'A')       = {norm_lev(s, s + 'A'):.3f}  (one ins)")
print(f"normLev('AAAA','TTTT')  = {norm_lev('AAAA', 'TTTT'):.3f}  (max-different)")

## 4. Load proteins (train70 + test30), filter by length and alphabet

Each row: `domain_id, aa_seq, ss_seq, SuperFamily`. We keep only proteins where length is in [50, 200] and the AA sequence uses only standard 20 letters. Build two dicts: `train_seqs` and `test_seqs`.

In [ ]:
def load_protein_csv(path):
    df = pd.read_csv(path, compression='gzip')
    print(f'  raw: {len(df)} rows, columns: {list(df.columns)}')
    df = df.dropna(subset=['aa_seq', 'SuperFamily'])
    df['aa_seq'] = df['aa_seq'].astype(str)
    df = df[df['aa_seq'].str.len().between(MIN_LEN_KEEP, MAX_LEN_KEEP)]
    df = df[df['aa_seq'].apply(is_standard)]
    print(f'  after length+alphabet filter: {len(df)} proteins')
    return df

print('Loading train70…')
train_df = load_protein_csv(os.path.join(DATA_DIR, 'cath_s20_train70.csv.gz'))
print('Loading test30…')
test_df = load_protein_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'))

# Build id → (aa_seq, SuperFamily) dicts
train_seqs = {row['domain_id']: row['aa_seq'] for _, row in train_df.iterrows()}
train_sf   = {row['domain_id']: row['SuperFamily'] for _, row in train_df.iterrows()}
test_seqs  = {row['domain_id']: row['aa_seq'] for _, row in test_df.iterrows()}
test_sf    = {row['domain_id']: row['SuperFamily'] for _, row in test_df.iterrows()}

print(f'\ntrain proteins kept: {len(train_seqs)}  test proteins kept: {len(test_seqs)}')
print(f'unique train superfamilies: {len(set(train_sf.values()))}')
print(f'unique test superfamilies:  {len(set(test_sf.values()))}')

# Length distribution sanity
train_lens = [len(s) for s in train_seqs.values()]
plt.figure(figsize=(8, 3))
plt.hist(train_lens, bins=30, edgecolor='k', alpha=0.75)
plt.xlabel('Sequence length'); plt.ylabel('Count')
plt.title(f'train70 length distribution after filtering ({len(train_seqs)} proteins)')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 5. Load natural pair samples

Two pair files:
- `cath_s20_pairs_sample.csv.gz` — 50K random pairs from the full pairwise file. **No header** (`shuf` strips it). Used for training (filtered to train_ids) and held-out V1 natural eval (filtered to test_ids).
- `cath_s20_pairs_high.csv.gz` — 6,765 high-similarity pairs (`aa_score ≥ 0.3`). **Has header.** Optional honest-eval set (real biology, high-sim regime).

Schema: `domain_p2, domain_p1, ss_score, aa_score, TM_min, TM_max, cath_superFamily`.

In [ ]:
PAIR_COLS = ['domain_p2', 'domain_p1', 'ss_score', 'aa_score',
             'TM_min', 'TM_max', 'cath_superFamily']

# Sample file: NO header (shuf'd from full file)
pairs_sample = pd.read_csv(
    os.path.join(DATA_DIR, 'cath_s20_pairs_sample.csv.gz'),
    compression='gzip', header=None, names=PAIR_COLS)
print(f'pairs_sample: {len(pairs_sample)} rows')
print(pairs_sample.head(3))

# High-sim file: HAS header
pairs_high = pd.read_csv(
    os.path.join(DATA_DIR, 'cath_s20_pairs_high.csv.gz'),
    compression='gzip')
print(f'\npairs_high: {len(pairs_high)} rows (header included)')
print(pairs_high.head(3))

# aa_score must be numeric (in case any row sneaks through as string)
pairs_sample['aa_score'] = pd.to_numeric(pairs_sample['aa_score'], errors='coerce')
pairs_sample = pairs_sample.dropna(subset=['aa_score'])
pairs_high['aa_score'] = pd.to_numeric(pairs_high['aa_score'], errors='coerce')
pairs_high = pairs_high.dropna(subset=['aa_score'])

print(f'\naa_score dist in pairs_sample:')
print(pairs_sample['aa_score'].describe())

## 6. Filter pair samples to train / held-out protein sets

Each pair (`domain_p1`, `domain_p2`) survives only if both IDs are in our filtered protein dict. Pairs with one side in train and the other in test are dropped (they'd leak).

In [ ]:
def filter_pairs_to_set(df, id_set):
    return df[df['domain_p1'].isin(id_set) & df['domain_p2'].isin(id_set)].reset_index(drop=True)

train_ids = set(train_seqs.keys())
test_ids  = set(test_seqs.keys())

natural_train_pairs = filter_pairs_to_set(pairs_sample, train_ids)
natural_test_pairs  = filter_pairs_to_set(pairs_sample, test_ids)
high_test_pairs     = filter_pairs_to_set(pairs_high,   test_ids)

print(f'natural train pairs (both ends in train70): {len(natural_train_pairs)}')
print(f'natural test pairs (both ends in test30):   {len(natural_test_pairs)}')
print(f'high-sim test pairs (both ends in test30):  {len(high_test_pairs)}')

# How does aa_score distribute in our train-side natural pairs?
plt.figure(figsize=(8, 3))
plt.hist(natural_train_pairs['aa_score'], bins=40, edgecolor='k', alpha=0.75)
plt.xlabel('aa_score (normalized AA Levenshtein similarity)'); plt.ylabel('Count')
plt.title(f'Natural train pairs aa_score distribution ({len(natural_train_pairs)} pairs)')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 7. Build training pair list — natural + synthetic mix

**Natural** pairs: take up to `N_NATURAL_TRAIN` from `natural_train_pairs`. Label = `aa_score`.

**Synthetic** pairs: pick a seed from `train_seqs`, perturb with k random sub/ins/del where k ∈ [1, 0.8·len(seed)]. Compute actual `norm_lev`. Skip pairs where the perturbed sequence exceeds MAX_LEN.

Why the synthetic side: natural CATH-S20 pairs have ~zero mass above 0.3 (S20 is redundancy-reduced). Without synthetic high-similarity coverage, the model never sees the [0.3, 1.0] range during training.

In [ ]:
N_NATURAL_TRAIN = 30000
N_SYNTH_TRAIN   = 30000
N_NATURAL_HELD  = 5000
N_SYNTH_HELD    = 5000

def perturb(seq, k, max_len=MAX_LEN):
    s = list(seq)
    for _ in range(k):
        if len(s) == 0:
            op = 'ins'
        elif len(s) >= max_len:
            op = rng.choice(['sub', 'del'])
        else:
            op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s))
            choices = [c for c in AA_ALPHABET if c != s[i]]
            s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1)
            s.insert(i, rng.choice(list(AA_ALPHABET)))
        elif op == 'del':
            i = rng.integers(0, len(s))
            del s[i]
    return ''.join(s)

def make_natural_pairs(natural_df, seq_dict, n_pairs):
    if len(natural_df) >= n_pairs:
        df = natural_df.sample(n=n_pairs, random_state=SEED).reset_index(drop=True)
    else:
        df = natural_df
        print(f'  warning: only {len(df)} natural pairs available, requested {n_pairs}')
    pairs = []
    for _, r in df.iterrows():
        a = seq_dict.get(r['domain_p1']); b = seq_dict.get(r['domain_p2'])
        if a is None or b is None:
            continue
        pairs.append((a, b, float(r['aa_score'])))
    return pairs

def make_synthetic_pairs(seq_dict, n_pairs):
    seeds = list(seq_dict.values())
    pairs = []
    attempts = 0
    while len(pairs) < n_pairs and attempts < n_pairs * 4:
        attempts += 1
        seed = seeds[int(rng.integers(0, len(seeds)))]
        k = int(rng.integers(1, max(2, int(0.8 * len(seed)))))
        other = perturb(seed, k)
        if len(other) < 1 or len(other) > MAX_LEN:
            continue
        label = norm_lev(seed, other)
        pairs.append((seed, other, label))
    return pairs

print('Building natural training pairs…')
train_natural = make_natural_pairs(natural_train_pairs, train_seqs, N_NATURAL_TRAIN)
print(f'  got {len(train_natural)} natural train pairs')

print('Building synthetic training pairs…')
train_synth = make_synthetic_pairs(train_seqs, N_SYNTH_TRAIN)
print(f'  got {len(train_synth)} synthetic train pairs')

print('Building natural held-out pairs…')
held_natural = make_natural_pairs(natural_test_pairs, test_seqs, N_NATURAL_HELD)
print(f'  got {len(held_natural)} natural held-out pairs')

print('Building synthetic held-out pairs…')
held_synth = make_synthetic_pairs(test_seqs, N_SYNTH_HELD)
print(f'  got {len(held_synth)} synthetic held-out pairs')

train_pairs = train_natural + train_synth
rng.shuffle(train_pairs)
print(f'\nTotal training pairs: {len(train_pairs)}')

# Combined-train label histogram
all_labels = np.array([p[2] for p in train_pairs])
plt.figure(figsize=(8, 3))
plt.hist([np.array([p[2] for p in train_natural]),
          np.array([p[2] for p in train_synth])],
         bins=40, stacked=True, label=['natural', 'synthetic'], edgecolor='k', alpha=0.75)
plt.xlabel('Label (normLev similarity)'); plt.ylabel('Count')
plt.title('Training-pair label distribution (natural + synthetic stacked)')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 8. Dataset & DataLoader

In [ ]:
class PairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        a, b, label = self.pairs[i]
        return (encode_pad(a), encode_pad(b),
                torch.tensor(label, dtype=torch.float32))

train_ds = PairDataset(train_pairs)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)

a, b, y = next(iter(train_dl))
print(f'Batch shapes: a={tuple(a.shape)}, b={tuple(b.shape)}, label={tuple(y.shape)}')
print(f'PAD count in first sequence: {(a[0] == PAD_IDX).sum().item()} / {MAX_LEN}')

## 9. Pad-aware Siamese encoder

Same architecture as colab10 with `MAX_LEN=200` and `torch.linalg.vector_norm` instead of deprecated `torch.norm`.

In [ ]:
class SiameseEncoder(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, hidden1=32, hidden2=64,
                 max_len=MAX_LEN, out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.fc = nn.Linear(hidden2 * max_len, out_dim)
    def forward(self, x):
        mask = (x != self.pad_idx).float()       # (B, L)
        x = self.embedding(x)                    # (B, L, embed_dim)
        x = x.permute(0, 2, 1)                   # (B, embed_dim, L)
        x = F.relu(self.conv1(x))                # (B, hidden1, L)
        x = F.relu(self.conv2(x))                # (B, hidden2, L)
        x = x * mask.unsqueeze(1)                # zero PAD activations
        x = x.flatten(1)                         # (B, hidden2 * L)
        x = self.fc(x)                           # (B, out_dim)
        x = F.normalize(x, p=2, dim=1)           # unit hypersphere
        return x

class SiameseNetwork(nn.Module):
    def __init__(self, **kw):
        super().__init__()
        self.encoder = SiameseEncoder(**kw)
    def forward(self, a, b):
        ea = self.encoder(a)
        eb = self.encoder(b)
        l2 = torch.linalg.vector_norm(ea - eb, ord=2, dim=1)
        sim = 1.0 - l2 / 2.0
        return sim
    def encode(self, x):
        return self.encoder(x)

model = SiameseNetwork().to(device)
print(model)
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

test_seq = encode_pad(list(train_seqs.values())[0]).unsqueeze(0).to(device)
with torch.no_grad():
    print(f'sim(seq, seq) before training = {model(test_seq, test_seq).item():.4f}  (expect 1.0)')

## 10. Training — plain MSE on normLev

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 30
losses = []

model.train()
for epoch in range(1, num_epochs + 1):
    epoch_loss = 0.0; nb = 0
    for a, b, label in train_dl:
        a = a.to(device); b = b.to(device); label = label.to(device)
        pred = model(a, b)
        loss = F.mse_loss(pred, label)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item(); nb += 1
    avg = epoch_loss / nb
    losses.append(avg)
    if epoch % 2 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{num_epochs} — MSE: {avg:.5f}')

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.title('Training Loss — plain MSE on mixed natural + synthetic pairs')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f'Final MSE: {losses[-1]:.5f}')

## 11. V1 evaluation — predicted vs true normLev

Two scatters side-by-side:
- **Synthetic held-out** — full label range [0, 1], tests calibration across the function.
- **Natural held-out** — real CATH biology; expect concentrated mass in [0.1, 0.3] (S20 distribution).

Plus the high-sim natural eval set on test30 (real biology in the high-sim regime, separate from training).

In [ ]:
def predict_pairs(pairs, batch_size=512):
    model.eval()
    true_v, pred_v = [], []
    with torch.no_grad():
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i+batch_size]
            a = torch.stack([encode_pad(p[0]) for p in batch]).to(device)
            b = torch.stack([encode_pad(p[1]) for p in batch]).to(device)
            pred = model(a, b).cpu().numpy()
            true_v.extend([p[2] for p in batch])
            pred_v.extend(pred.tolist())
    return np.array(true_v), np.array(pred_v)

# High-sim natural pairs as triples (a, b, aa_score)
high_pairs_triples = []
for _, r in high_test_pairs.iterrows():
    a = test_seqs.get(r['domain_p1']); b = test_seqs.get(r['domain_p2'])
    if a and b:
        high_pairs_triples.append((a, b, float(r['aa_score'])))
print(f'High-sim natural eval pairs: {len(high_pairs_triples)}')

true_synth, pred_synth = predict_pairs(held_synth)
true_nat,   pred_nat   = predict_pairs(held_natural)
true_high,  pred_high  = (np.array([]), np.array([]))
if len(high_pairs_triples) > 10:
    true_high, pred_high = predict_pairs(high_pairs_triples)

def report(name, true_v, pred_v):
    if len(true_v) < 10:
        print(f'{name}: too few pairs ({len(true_v)})'); return
    pr, _ = pearsonr(true_v, pred_v)
    sr, _ = spearmanr(true_v, pred_v)
    mask = true_v > 0.5
    sr_high = spearmanr(true_v[mask], pred_v[mask])[0] if mask.sum() > 10 else float('nan')
    print(f'{name}: n={len(true_v):>5}  Pearson={pr:.3f}  Spearman={sr:.3f}  Spearman(>0.5)={sr_high:.3f}')
    return pr, sr, sr_high

print('\n=== V1 metrics ===')
report('synthetic held-out', true_synth, pred_synth)
report('natural   held-out', true_nat,   pred_nat)
report('high-sim  held-out', true_high,  pred_high)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (true_v, pred_v, title) in zip(axes, [
    (true_synth, pred_synth, 'Synthetic held-out'),
    (true_nat,   pred_nat,   'Natural held-out'),
    (true_high,  pred_high,  'High-sim natural held-out'),
]):
    if len(true_v) < 10:
        ax.set_title(f'{title}\n(too few pairs)'); continue
    ax.scatter(true_v, pred_v, alpha=0.15, s=6)
    ax.plot([0, 1], [0, 1], 'r-', linewidth=2, label='y = x')
    pr, _ = pearsonr(true_v, pred_v)
    sr, _ = spearmanr(true_v, pred_v)
    ax.set_title(f'{title}\nn={len(true_v)}  r={pr:.3f}  ρ={sr:.3f}')
    ax.set_xlabel('True normLev'); ax.set_ylabel('Predicted')
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 12. V2 — same vs different SuperFamily separation

Sample random pairs of test30 proteins, predict similarity, compare distributions for same-superfamily vs different-superfamily pairs. Report AUROC where the positive label is `same superfamily = 1`.

Success criterion (supervisor): same-superfamily pairs cluster high, different-superfamily pairs cluster low.

In [ ]:
N_V2_PAIRS = 5000
test_id_list = list(test_seqs.keys())

v2_pairs = []
v2_labels = []   # 1 if same SuperFamily else 0
while len(v2_pairs) < N_V2_PAIRS:
    i, j = rng.integers(0, len(test_id_list), size=2)
    if i == j:
        continue
    id_a, id_b = test_id_list[i], test_id_list[j]
    same = int(test_sf[id_a] == test_sf[id_b])
    v2_pairs.append((test_seqs[id_a], test_seqs[id_b], 0.0))   # dummy label, unused
    v2_labels.append(same)

_, v2_pred = predict_pairs(v2_pairs)
v2_labels = np.array(v2_labels)

n_same = int(v2_labels.sum())
n_diff = int((1 - v2_labels).sum())
print(f'V2 sampled pairs: {len(v2_pred)}  (same SF: {n_same}, different SF: {n_diff})')

if n_same > 5 and n_diff > 5:
    auroc = roc_auc_score(v2_labels, v2_pred)
    print(f'V2 AUROC (same vs different SuperFamily): {auroc:.3f}')
    plt.figure(figsize=(9, 5))
    plt.hist(v2_pred[v2_labels == 0], bins=40, alpha=0.6, label=f'different SF (n={n_diff})', edgecolor='k')
    plt.hist(v2_pred[v2_labels == 1], bins=40, alpha=0.6, label=f'same SF (n={n_same})', edgecolor='k')
    plt.xlabel('Predicted similarity'); plt.ylabel('Count')
    plt.title(f'V2: predicted similarity by SuperFamily class — AUROC={auroc:.3f}')
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
else:
    print('V2: too few same-SuperFamily pairs in random sample. Increase N_V2_PAIRS or use stratified sampling.')

## 13. Decision criteria

**Iter-1 success bar (real-CATH baseline):**
- Synthetic held-out V1: Pearson r > 0.85 (matches colab10's synthetic-only result).
- Natural held-out V1: Pearson r > 0.5 — natural pairs are concentrated in [0.1, 0.3], so absolute r will be lower than synthetic.
- V2 AUROC > 0.7 — indicates same-superfamily pairs are predicted higher than different-superfamily pairs more often than chance.
- High-sim held-out V1: visually shows points landing in the upper right, even if scatter is wide.

**If iter 1 passes:**
- Move to Phase 5 (cross-representation eval). Load `cath_aa_ss_seq_superfamily_dssp_3di.csv.gz`, look up 3Di sequences for test30 IDs, run trained model on 3Di inputs (vocab is shared because both alphabets are size 20 + PAD = 21). Compare V1 and V2 against AA results.
- Then train a sibling model on 3Di and run reverse direction (3Di-trained on AA) for the bidirectional robustness test.

**If iter 1 fails:**
- If natural V1 looks flat (constant prediction): increase synthetic share or oversample low-natural pairs (the model is collapsing on the easier synthetic distribution).
- If V2 AUROC ~0.5: model is not learning biologically meaningful structure. Check that train and test proteins genuinely differ; try richer encoder (`embed_dim=64, out_dim=256`).
- If synthetic V1 also poor: regression on architecture — go back to colab10 settings and verify the loader didn't break something.